In [ ]:
!pip -q install pystac-client planetary-computer geopandas rasterio shapely pyproj

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 570.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 1.6 MB/s eta 0:00:00


In [ ]:
from google.colab import files

uploaded = files.upload()

AOI_PATH = next(iter(uploaded.keys()))
print("AOI:", AOI_PATH)

Saving HSTDNNXuanThuy.geojson to HSTDNNXuanThuy.geojson
AOI: HSTDNNXuanThuy.geojson


In [ ]:
import geopandas as gpd
from shapely.ops import unary_union

aoi = gpd.read_file(AOI_PATH)

print("Số feature:", len(aoi))
print("CRS:", aoi.crs)
print("Bounds:", aoi.total_bounds)

if aoi.crs is None:
    raise ValueError("AOI không có CRS.")

# CRS đầu ra phải khớp AOI
TARGET_CRS = aoi.crs

# Gộp toàn bộ 568 feature thành 1 AOI
aoi_union = unary_union(aoi.geometry)

print("Target CRS:", TARGET_CRS)
print("Geometry:", aoi_union.geom_type)

Số feature: 568
CRS: EPSG:32648
Bounds: [ 650503.5977 2231814.6382  668704.6741 2245549.0359]
Target CRS: EPSG:32648
Geometry: Polygon


In [ ]:
from shapely.geometry import mapping
from shapely.ops import unary_union

aoi_4326 = aoi.to_crs("EPSG:4326")
aoi_search_geom = unary_union(aoi_4326.geometry)

search_geometry = mapping(aoi_search_geom)

print("Bounds EPSG:4326:")
print(aoi_4326.total_bounds)

Bounds EPSG:4326:
[106.44089698  20.1773699  106.6149193   20.30106291]


In [ ]:
from pystac_client import Client
import planetary_computer

catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)

print("Connected to Planetary Computer")

Connected to Planetary Computer


In [ ]:
YEAR = "2026"

search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=search_geometry,
    datetime=f"{YEAR}-01-01T00:00:00Z/{YEAR}-12-31T23:59:59Z"
)

items = list(search.items())

print(f"Tìm thấy {len(items)} Sentinel-2 item(s) trong năm {YEAR}:")
for item in items:
    print(f"- {item.id} ({item.datetime})")

Tìm thấy 127 Sentinel-2 item(s) trong năm 2026:
- S2B_MSIL2A_20260929T032539_R018_T48QXH_20260929T071432 (2026-09-29 03:25:39.024000+00:00)
- S2B_MSIL2A_20260926T031539_R118_T48QXH_20260926T070812 (2026-09-26 03:15:39.024000+00:00)
- S2C_MSIL2A_20260924T032521_R018_T48QXH_20260924T082855 (2026-09-24 03:25:21.025000+00:00)
- S2A_MSIL2A_20260923T032151_R118_T48QXH_20260923T065854 (2026-09-23 03:21:51.024000+00:00)
- S2C_MSIL2A_20260921T031521_R118_T48QXH_20260921T082912 (2026-09-21 03:15:21.025000+00:00)
- S2B_MSIL2A_20260919T032539_R018_T48QXH_20260919T072856 (2026-09-19 03:25:39.024000+00:00)
- S2A_MSIL2A_20260916T033151_R018_T48QXH_20260916T070354 (2026-09-16 03:31:51.024000+00:00)
- S2B_MSIL2A_20260916T031539_R118_T48QXH_20260916T071312 (2026-09-16 03:15:39.024000+00:00)
- S2C_MSIL2A_20260914T032511_R018_T48QXH_20260914T082918 (2026-09-14 03:25:11.025000+00:00)
- S2C_MSIL2A_20260911T031521_R118_T48QXH_20260911T082208 (2026-09-11 03:15:21.025000+00:00)
- S2B_MSIL2A_20260909T032529_R01

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape

if len(items) == 0:
    raise RuntimeError(
        f"Không tìm thấy Sentinel-2 L2A giao với AOI trong năm {YEAR}."
    )

results = []

for item in items:
    item_geom_4326 = shape(item.geometry)

    item_geom_target = (
        gpd.GeoSeries(
            [item_geom_4326],
            crs="EPSG:4326"
        )
        .to_crs(TARGET_CRS)
        .iloc[0]
    )

    intersection = item_geom_target.intersection(aoi_union)

    # Tính phần trăm diện tích AOI được ảnh này bao phủ
    coverage = (
        intersection.area / aoi_union.area * 100
        if aoi_union.area > 0
        else 0
    )

    results.append({
        "id": item.id,
        "datetime": item.datetime.strftime("%Y-%m-%d %H:%M:%S"),
        "mgrs_tile": item.properties.get("s2:mgrs_tile"),
        "cloud_cover": item.properties.get("eo:cloud_cover"),
        "coverage_AOI_percent": round(coverage, 4)
    })

df = pd.DataFrame(results)

# Sắp xếp theo tỷ lệ bao phủ giảm dần và độ mây tăng dần
df = df.sort_values(
    ["coverage_AOI_percent", "cloud_cover"],
    ascending=[False, True]
)

# Kiểm tra xem có ảnh nào phủ 100% AOI không
full_cover_df = df[df["coverage_AOI_percent"] >= 99.9]
partial_cover_df = df[df["coverage_AOI_percent"] < 99.9]

print(f"=== KẾT QUẢ KIỂM TRA ĐỘ PHỦ AOI ===")
print(f"Tổng số ảnh tìm thấy: {len(df)}")
print(f"- Số ảnh PHỦ KÍN AOI (>= 99.9%): {len(full_cover_df)}")
print(f"- Số ảnh CHỈ PHỦ MỘT PHẦN AOI (< 99.9%): {len(partial_cover_df)}")
print("\n* Lưu ý: Với các ảnh chỉ phủ một phần, khi cắt (clip) đơn lẻ sẽ bị khuyết dữ liệu ở các vùng ngoài biên ảnh.")
print("Để lấy dữ liệu trọn vẹn từ các ảnh này, chúng ta cần gộp (mosaic) chúng lại trước khi cắt.\n")

display(df.head(20))

=== KẾT QUẢ KIỂM TRA ĐỘ PHỦ AOI ===
Tổng số ảnh tìm thấy: 127
- Số ảnh PHỦ KÍN AOI (>= 99.9%): 62
- Số ảnh CHỈ PHỦ MỘT PHẦN AOI (< 99.9%): 65

* Lưu ý: Với các ảnh chỉ phủ một phần, khi cắt (clip) đơn lẻ sẽ bị khuyết dữ liệu ở các vùng ngoài biên ảnh.
Để lấy dữ liệu trọn vẹn từ các ảnh này, chúng ta cần gộp (mosaic) chúng lại trước khi cắt.



,id,datetime,mgrs_tile,cloud_cover,coverage_AOI_percent
62,S2A_MSIL2A_20260526T032151_R118_T48QXH_2026052...,2026-05-26 03:21:51,48QXH,0.010129,100.0
37,S2B_MSIL2A_20260718T031539_R118_T48QXH_2026071...,2026-07-18 03:15:39,48QXH,0.853850,100.0
4,S2C_MSIL2A_20260921T031521_R118_T48QXH_2026092...,2026-09-21 03:15:21,48QXH,1.965456,100.0
123,S2B_MSIL2A_20260109T032019_R118_T48QXH_2026010...,2026-01-09 03:20:19,48QXH,2.069157,100.0
24,S2C_MSIL2A_20260812T031541_R118_T48QXH_2026081...,2026-08-12 03:15:41,48QXH,3.009039,100.0
13,S2A_MSIL2A_20260903T032151_R118_T48QXH_2026090...,2026-09-03 03:21:51,48QXH,4.914029,100.0
121,S2C_MSIL2A_20260114T032111_R118_T48QXH_2026011...,2026-01-14 03:21:11,48QXH,17.774290,100.0
63,S2C_MSIL2A_20260524T031541_R118_T48QXH_2026052...,2026-05-24 03:15:41,48QXH,18.238774,100.0
9,S2C_MSIL2A_20260911T031521_R118_T48QXH_2026091...,2026-09-11 03:15:21,48QXH,27.286136,100.0
113,S2C_MSIL2A_20260203T031931_R118_T48QXH_2026020...,2026-02-03 03:19:31,48QXH,27.508244,100.0


In [ ]:
item_info = []

for item in items:
    geom_target = (
        gpd.GeoSeries(
            [shape(item.geometry)],
            crs="EPSG:4326"
        )
        .to_crs(TARGET_CRS)
        .iloc[0]
    )

    coverage = (
        geom_target.intersection(aoi_union).area
        / aoi_union.area
    )

    cloud = item.properties.get("eo:cloud_cover")

    if cloud is None:
        cloud = 999

    # Lọc ảnh có mây < 10% VÀ phải cover 100% (>= 99.9%) diện tích AOI
    if cloud < 10 and coverage >= 0.999:
        item_info.append({
            "item": item,
            "coverage": coverage,
            "cloud": cloud
        })

print(f"Tìm thấy {len(item_info)} ảnh có độ phủ mây < 10% và bao phủ 100% AOI.")

# Sắp xếp các ảnh theo độ phủ mây tăng dần
item_info.sort(key=lambda x: x["cloud"])

# Gán toàn bộ các ảnh thỏa mãn điều kiện trên vào selected_items
selected_items = [x["item"] for x in item_info]

print(f"Đã gán {len(selected_items)} ảnh này vào danh sách `selected_items`.")
for idx, x in enumerate(item_info, 1):
    print(f"{idx}. {x['item'].id} - Cloud: {round(x['cloud'], 4)}% - Coverage: {round(x['coverage']*100, 2)}%")

Tìm thấy 6 ảnh có độ phủ mây < 10% và bao phủ 100% AOI.
Đã gán 6 ảnh này vào danh sách `selected_items`.
1. S2A_MSIL2A_20260526T032151_R118_T48QXH_20260526T075520 - Cloud: 0.0101% - Coverage: 100.0%
2. S2B_MSIL2A_20260718T031539_R118_T48QXH_20260718T070226 - Cloud: 0.8538% - Coverage: 100.0%
3. S2C_MSIL2A_20260921T031521_R118_T48QXH_20260921T082912 - Cloud: 1.9655% - Coverage: 100.0%
4. S2B_MSIL2A_20260109T032019_R118_T48QXH_20260109T070213 - Cloud: 2.0692% - Coverage: 100.0%
5. S2C_MSIL2A_20260812T031541_R118_T48QXH_20260812T082613 - Cloud: 3.009% - Coverage: 100.0%
6. S2A_MSIL2A_20260903T032151_R118_T48QXH_20260903T075409 - Cloud: 4.914% - Coverage: 100.0%


In [ ]:
item = selected_items[0]

print("Available assets:")
print(list(item.assets.keys()))

Available assets:
['AOT', 'B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B11', 'B12', 'B8A', 'SCL', 'WVP', 'visual', 'safe-manifest', 'granule-metadata', 'inspire-metadata', 'product-metadata', 'datastrip-metadata', 'tilejson', 'rendered_preview']


In [ ]:
BANDS = [
    "B02", "B03", "B04",  # Blue, Green, Red (10m)
    "B8A",                # NIR Narrow (20m)
    "B11", "B12"          # SWIR 1, 2 (20m)
]

TARGET_RESOLUTION = 10
print(f"Cấu hình mới: {len(BANDS)} bands ở độ phân giải {TARGET_RESOLUTION}m")

Cấu hình mới: 6 bands ở độ phân giải 10m


In [ ]:
import math
from rasterio.transform import from_origin

minx, miny, maxx, maxy = aoi_union.bounds

res = TARGET_RESOLUTION

# Snap grid vào bội số 10 m
left   = math.floor(minx / res) * res
bottom = math.floor(miny / res) * res
right  = math.ceil(maxx / res) * res
top    = math.ceil(maxy / res) * res

width = int(round((right - left) / res))
height = int(round((top - bottom) / res))

target_transform = from_origin(
    left,
    top,
    res,
    res
)

print("CRS:", TARGET_CRS)
print("Resolution:", res, "m")
print("Width:", width)
print("Height:", height)
print("Bounds:", (left, bottom, right, top))
print("Transform:")
print(target_transform)

CRS: EPSG:32648
Resolution: 10 m
Width: 1821
Height: 1374
Bounds: (650500, 2231810, 668710, 2245550)
Transform:
| 10.00, 0.00, 650500.00|
| 0.00,-10.00, 2245550.00|
| 0.00, 0.00, 1.00|


In [ ]:
from rasterio.features import geometry_mask
from shapely.geometry import mapping

inside_aoi = geometry_mask(
    [mapping(aoi_union)],
    out_shape=(height, width),
    transform=target_transform,
    invert=True
)

print("Pixel bên trong AOI:", inside_aoi.sum())

Pixel bên trong AOI: 1471400


In [ ]:
import numpy as np
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

# Khởi tạo stack cho 10 bands
stack = np.zeros(
    (len(BANDS), height, width),
    dtype=np.uint16
)

for band_idx, band in enumerate(BANDS):
    print(f"\nĐang xử lý kênh {band} ...")
    mosaic = np.zeros((height, width), dtype=np.uint16)
    filled = np.zeros((height, width), dtype=bool)

    for item in selected_items:
        if band not in item.assets:
            continue

        href = item.assets[band].href
        with rasterio.open(href) as src:
            # Sử dụng WarpedVRT để tự động resample các band 20m lên 10m
            with WarpedVRT(
                src,
                crs=TARGET_CRS,
                transform=target_transform,
                width=width,
                height=height,
                resampling=Resampling.bilinear,
                src_nodata=0,
                nodata=0
            ) as vrt:
                arr = vrt.read(1)

        valid = ((arr != 0) & inside_aoi & (~filled))
        mosaic[valid] = arr[valid]
        filled[valid] = True

    mosaic[~inside_aoi] = 0
    stack[band_idx] = mosaic
    print(f"  Hoàn thành {band}.")


Đang xử lý kênh B02 ...
  Hoàn thành B02.

Đang xử lý kênh B03 ...
  Hoàn thành B03.

Đang xử lý kênh B04 ...
  Hoàn thành B04.

Đang xử lý kênh B8A ...
  Hoàn thành B8A.

Đang xử lý kênh B11 ...
  Hoàn thành B11.

Đang xử lý kênh B12 ...
  Hoàn thành B12.


In [ ]:
import numpy as np
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

# Lặp qua tất cả các ảnh đã được chọn
for idx, item in enumerate(selected_items, start=1):
    image_date = item.datetime.strftime("%Y-%m-%d")
    output_filename = f"Sentinel2_{image_date}_{item.id[:20]}_HSTDNNXUANTHUY_6bands_10m.tif"

    # Khởi tạo mảng stack chứa 6 kênh cho ảnh hiện tại
    current_stack = np.zeros(
        (len(BANDS), height, width),
        dtype=np.uint16
    )

    print(f"\nProcessing item {idx}/{len(selected_items)}: {item.id} ({image_date})")

    for band_idx, band in enumerate(BANDS):
        if band not in item.assets:
            print(f"  Kênh {band} không tồn tại trong item, bỏ qua.")
            continue

        href = item.assets[band].href
        with rasterio.open(href) as src:
            # Sử dụng WarpedVRT để tự động resample và align khớp lưới tọa độ chuẩn
            with WarpedVRT(
                src,
                crs=TARGET_CRS,
                transform=target_transform,
                width=width,
                height=height,
                resampling=Resampling.bilinear,
                src_nodata=0,
                nodata=0
            ) as vrt:
                arr = vrt.read(1)

        # Mask theo AOI
        arr[~inside_aoi] = 0
        current_stack[band_idx] = arr

    # Cấu hình thuộc tính cho file GeoTIFF đầu ra
    profile = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": len(BANDS),
        "dtype": "uint16",
        "crs": TARGET_CRS,
        "transform": target_transform,
        "nodata": 0,
        "compress": "deflate",
        "tiled": True
    }

    # Lưu ảnh ra file GeoTIFF
    with rasterio.open(output_filename, "w", **profile) as dst:
        for i, band in enumerate(BANDS, start=1):
            dst.write(current_stack[i - 1], i)
            dst.set_band_description(i, band)

        dst.update_tags(
            acquisition_date=image_date,
            source="Microsoft Planetary Computer - Sentinel-2 L2A",
            bands=",".join(BANDS)
        )

    print(f"  -> Saved: {output_filename}")


Processing item 1/6: S2A_MSIL2A_20260526T032151_R118_T48QXH_20260526T075520 (2026-05-26)
  -> Saved: Sentinel2_2026-05-26_S2A_MSIL2A_20260526T_HSTDNNXUANTHUY_6bands_10m.tif

Processing item 2/6: S2B_MSIL2A_20260718T031539_R118_T48QXH_20260718T070226 (2026-07-18)
  -> Saved: Sentinel2_2026-07-18_S2B_MSIL2A_20260718T_HSTDNNXUANTHUY_6bands_10m.tif

Processing item 3/6: S2C_MSIL2A_20260921T031521_R118_T48QXH_20260921T082912 (2026-09-21)
  -> Saved: Sentinel2_2026-09-21_S2C_MSIL2A_20260921T_HSTDNNXUANTHUY_6bands_10m.tif

Processing item 4/6: S2B_MSIL2A_20260109T032019_R118_T48QXH_20260109T070213 (2026-01-09)
  -> Saved: Sentinel2_2026-01-09_S2B_MSIL2A_20260109T_HSTDNNXUANTHUY_6bands_10m.tif

Processing item 5/6: S2C_MSIL2A_20260812T031541_R118_T48QXH_20260812T082613 (2026-08-12)
  -> Saved: Sentinel2_2026-08-12_S2C_MSIL2A_20260812T_HSTDNNXUANTHUY_6bands_10m.tif

Processing item 6/6: S2A_MSIL2A_20260903T032151_R118_T48QXH_20260903T075409 (2026-09-03)
  -> Saved: Sentinel2_2026-09-03_S2A_MSI

In [ ]:
image_date = selected_items[0].datetime.strftime("%Y-%m-%d")
OUTPUT_STACK_6B = f"Sentinel2_Mosaic_{YEAR}_HSTDNNXUANTHUY_6bands_10m.tif"

profile.update({
    "count": len(BANDS),
    "description": "Sentinel-2 6-band mosaic stack (Blue, Green, Red, B8A, SWIR1, SWIR2) filtered for cloud < 10%"
})

with rasterio.open(OUTPUT_STACK_6B, "w", **profile) as dst:
    for i, band in enumerate(BANDS, start=1):
        dst.write(stack[i - 1], i)
        dst.set_band_description(i, band)

    dst.update_tags(
        acquisition_year=YEAR,
        bands_included=",".join(BANDS),
        description="Sentinel-2 6-band cloud-free mosaic over Xuan Thuy AOI"
    )

print(f"Đã lưu thành công tệp mosaic {len(BANDS)} bands: {OUTPUT_STACK_6B}")

Đã lưu thành công tệp mosaic 6 bands: Sentinel2_Mosaic_2026_HSTDNNXUANTHUY_6bands_10m.tif


In [ ]:
import rasterio
import glob

# Tìm tất cả các file ảnh tiff đã xuất
tif_files = sorted(glob.glob("*.tif"))

print("=== KIỂM TRA THÔNG TIN ĐỊA LÝ VÀ THỜI GIAN CỦA CÁC FILE XUẤT RA ===\n")

for f_path in tif_files:
    with rasterio.open(f_path) as src:
        meta_tags = src.tags() # Đọc thông tin thẻ metadata đi kèm
        print(f"File: {f_path}")
        print(f"  - Hệ tọa độ (CRS): {src.crs}")
        print(f"  - Kích thước (WxH): {src.width}x{src.height} pixels")
        print(f"  - Số kênh ảnh (Bands): {src.count}")
        print(f"  - Tọa độ góc trái trên: {src.transform[2]}, {src.transform[5]}")

        # Hiển thị thông tin thời gian thu nhận được lưu trong tags
        date_tag = meta_tags.get("acquisition_date") or meta_tags.get("acquisition_year")
        print(f"  - Thời gian thu nhận (Time/Year): {date_tag}")
        print(f"  - Danh sách kênh: {meta_tags.get('bands_included') or meta_tags.get('bands')}")
        print("-" * 50)

=== KIỂM TRA THÔNG TIN ĐỊA LÝ VÀ THỜI GIAN CỦA CÁC FILE XUẤT RA ===

File: Sentinel2_2026-01-09_S2B_MSIL2A_20260109T_HSTDNNXUANTHUY_6bands_10m.tif
  - Hệ tọa độ (CRS): EPSG:32648
  - Kích thước (WxH): 1821x1374 pixels
  - Số kênh ảnh (Bands): 6
  - Tọa độ góc trái trên: 650500.0, 2245550.0
  - Thời gian thu nhận (Time/Year): 2026-01-09
  - Danh sách kênh: B02,B03,B04,B8A,B11,B12
--------------------------------------------------
File: Sentinel2_2026-05-26_S2A_MSIL2A_20260526T_HSTDNNXUANTHUY_6bands_10m.tif
  - Hệ tọa độ (CRS): EPSG:32648
  - Kích thước (WxH): 1821x1374 pixels
  - Số kênh ảnh (Bands): 6
  - Tọa độ góc trái trên: 650500.0, 2245550.0
  - Thời gian thu nhận (Time/Year): 2026-05-26
  - Danh sách kênh: B02,B03,B04,B8A,B11,B12
--------------------------------------------------
File: Sentinel2_2026-07-18_S2B_MSIL2A_20260718T_HSTDNNXUANTHUY_6bands_10m.tif
  - Hệ tọa độ (CRS): EPSG:32648
  - Kích thước (WxH): 1821x1374 pixels
  - Số kênh ảnh (Bands): 6
  - Tọa độ góc trái trên: 6

In [ ]:
import zipfile
import glob
import os
from google.colab import files

# Định nghĩa tên tệp ZIP đầu ra
zip_filename = f"Sentinel2_XuanThuy_{YEAR}_6bands_10m.zip"

# Tìm tất cả các tệp .tif trong thư mục hiện tại
tif_files = glob.glob("*.tif")

if not tif_files:
    print("Không tìm thấy tệp .tif nào để nén!")
else:
    print(f"Tìm thấy {len(tif_files)} tệp ảnh. Bắt đầu nén vào {zip_filename}...")

    # Tiến hành đóng gói và nén
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file in tif_files:
            zipf.write(file, os.path.basename(file))
            print(f" -> Đã thêm: {file}")

    print(f"\nNén thành công! Đang chuẩn bị tải xuống tệp {zip_filename}...")

    # Trình duyệt sẽ tự động kích hoạt tải xuống tệp zip
    files.download(zip_filename)

Tìm thấy 7 tệp ảnh. Bắt đầu nén vào Sentinel2_XuanThuy_2026_6bands_10m.zip...
 -> Đã thêm: Sentinel2_2026-08-12_S2C_MSIL2A_20260812T_HSTDNNXUANTHUY_6bands_10m.tif
 -> Đã thêm: Sentinel2_2026-05-26_S2A_MSIL2A_20260526T_HSTDNNXUANTHUY_6bands_10m.tif
 -> Đã thêm: Sentinel2_2026-09-03_S2A_MSIL2A_20260903T_HSTDNNXUANTHUY_6bands_10m.tif
 -> Đã thêm: Sentinel2_2026-09-21_S2C_MSIL2A_20260921T_HSTDNNXUANTHUY_6bands_10m.tif
 -> Đã thêm: Sentinel2_2026-07-18_S2B_MSIL2A_20260718T_HSTDNNXUANTHUY_6bands_10m.tif
 -> Đã thêm: Sentinel2_2026-01-09_S2B_MSIL2A_20260109T_HSTDNNXUANTHUY_6bands_10m.tif
 -> Đã thêm: Sentinel2_Mosaic_2026_HSTDNNXUANTHUY_6bands_10m.tif

Nén thành công! Đang chuẩn bị tải xuống tệp Sentinel2_XuanThuy_2026_6bands_10m.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>